# 🧠 Daily Challenge: Classification with Neural Networks in TensorFlow

**What you'll build:**
- Binary classification NN on a circles dataset
- Decision boundary visualizations
- Progressive model improvements (layers, activations, optimizers)
- Train/test evaluation pipeline

> **Runtime tip:** Runtime → Change runtime type → **T4 GPU** for faster training.

---
## 1️⃣ Understand Classification Types

Before writing any code, let's understand the three classification paradigms.

### Binary Classification
The output belongs to **one of exactly two classes** (0 or 1).  
The output layer uses a **single neuron with Sigmoid** activation, producing a probability in [0, 1].  
**Loss function:** Binary Cross-Entropy.  
**Example:** Email spam detection — *spam* vs *not spam*.

### Multi-class Classification
The output belongs to **one of N mutually exclusive classes** (exactly one class per sample).  
The output layer uses **N neurons with Softmax** activation (probabilities sum to 1).  
**Loss function:** Categorical Cross-Entropy (or Sparse Categorical Cross-Entropy).  
**Example:** Handwritten digit recognition — one of {0, 1, 2, ..., 9}.

### Multi-label Classification
Each sample can belong to **multiple classes simultaneously** — labels are NOT mutually exclusive.  
The output layer uses **N neurons with Sigmoid** (each neuron independently predicts 0 or 1).  
**Loss function:** Binary Cross-Entropy applied per label.  
**Example:** Movie genre tagging — a film can be *Action*, *Comedy*, AND *Romance* at the same time.

| | Binary | Multi-class | Multi-label |
|---|---|---|---|
| Output neurons | 1 | N | N |
| Output activation | Sigmoid | Softmax | Sigmoid |
| Labels per sample | 1 | 1 | ≥ 1 |
| Loss | Binary CE | Categorical CE | Binary CE |

---
## 2️⃣ Set Up Environment & Create Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

print(f'TensorFlow : {tf.__version__}')
print(f'NumPy      : {np.__version__}')
print(f'GPU devices: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# ── Generate the circles dataset ──────────────────────────────────────────────
samples = 1000
X, y = make_circles(samples,
                    noise=0.03,
                    random_state=42)

print('X shape :', X.shape)   # (1000, 2) — two features: x₁ and x₂
print('y shape :', y.shape)   # (1000,)   — binary label 0 or 1
print('\nFirst 5 rows of X:\n', X[:5])
print('\nFirst 5 labels y:', y[:5])

In [ ]:
# ── Quick look at the data in a DataFrame ─────────────────────────────────────
df = pd.DataFrame({'x1': X[:, 0], 'x2': X[:, 1], 'label': y})
print(df.head(10))
print('\nClass distribution:')
print(df['label'].value_counts())

In [ ]:
# ── Visualize the dataset ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: coloured scatter
axes[0].scatter(X[y == 0, 0], X[y == 0, 1],
                c='steelblue', label='Class 0 (outer)', alpha=0.6, edgecolors='white', s=40)
axes[0].scatter(X[y == 1, 0], X[y == 1, 1],
                c='tomato',    label='Class 1 (inner)', alpha=0.6, edgecolors='white', s=40)
axes[0].set_title('Circles Dataset — both classes', fontweight='bold')
axes[0].set_xlabel('Feature x₁')
axes[0].set_ylabel('Feature x₂')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right: class distribution bar
counts = np.bincount(y)
axes[1].bar(['Class 0 (outer)', 'Class 1 (inner)'], counts,
            color=['steelblue', 'tomato'], edgecolor='white')
for i, c in enumerate(counts):
    axes[1].text(i, c + 5, str(c), ha='center', fontweight='bold')
axes[1].set_title('Class Distribution', fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('\nObservation: the two classes form concentric rings — a linear model CANNOT separate them.')
print('We need a neural network to learn the curved decision boundary.')

---
## 3️⃣ Build a Basic Neural Network (1 Dense Layer)

In [ ]:
tf.random.set_seed(42)

# ── Model 1: single dense layer, no hidden activation ─────────────────────────
model_1 = tf.keras.Sequential([
    tf.keras.layers.Dense(1, activation='sigmoid', input_shape=(2,))
], name='model_1_basic')

model_1.compile(
    loss='binary_crossentropy',
    optimizer='sgd',
    metrics=['accuracy']
)

model_1.summary()

In [ ]:
# ── Train Model 1 ─────────────────────────────────────────────────────────────
history_1 = model_1.fit(
    X, y,
    epochs=50,
    batch_size=32,
    verbose=0
)

loss_1, acc_1 = model_1.evaluate(X, y, verbose=0)
print(f'Model 1 — Loss: {loss_1:.4f}  |  Accuracy: {acc_1*100:.2f}%')
print('\nExpected: ~50% — a single linear layer cannot capture the circular boundary.')

---
## 4️⃣ Improve the Model (More Layers, More Neurons, Adam Optimizer)

In [ ]:
tf.random.set_seed(42)

# ── Model 2: two hidden layers + Adam ─────────────────────────────────────────
model_2 = tf.keras.Sequential([
    tf.keras.layers.Dense(16, input_shape=(2,)),   # hidden layer 1 (no activation yet)
    tf.keras.layers.Dense(16),                     # hidden layer 2
    tf.keras.layers.Dense(1, activation='sigmoid') # output
], name='model_2_improved')

model_2.compile(
    loss='binary_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    metrics=['accuracy']
)

model_2.summary()

In [ ]:
history_2 = model_2.fit(
    X, y,
    epochs=100,
    batch_size=32,
    verbose=0
)

loss_2, acc_2 = model_2.evaluate(X, y, verbose=0)
print(f'Model 2 — Loss: {loss_2:.4f}  |  Accuracy: {acc_2*100:.2f}%')

# ── Compare Models 1 vs 2 ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history_1.history['loss'],     label='Model 1 loss')
axes[0].plot(history_2.history['loss'][:50],label='Model 2 loss (first 50 ep)')
axes[0].set_title('Training Loss Comparison')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Cross-Entropy')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history_1.history['accuracy'],      label='Model 1 accuracy')
axes[1].plot(history_2.history['accuracy'][:50], label='Model 2 accuracy (first 50 ep)')
axes[1].set_title('Training Accuracy Comparison')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5️⃣ Visualize Decision Boundaries

In [ ]:
def plot_decision_boundary(model, X, y, title='Decision Boundary', ax=None):
    """
    Visualize where a model draws the line between classes.

    Steps:
      1. Create a dense mesh grid that covers the feature space.
      2. Get model predictions for every grid point.
      3. Colour the background by predicted class (contourf).
      4. Overlay the actual data points as a scatter plot.
    """
    # Grid resolution
    margin   = 0.15
    x_min, x_max = X[:, 0].min() - margin, X[:, 0].max() + margin
    y_min, y_max = X[:, 1].min() - margin, X[:, 1].max() + margin
    h = 0.01  # step size

    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, h),
        np.arange(y_min, y_max, h)
    )

    # Predict on every grid point
    grid_input = np.c_[xx.ravel(), yy.ravel()]
    preds      = model.predict(grid_input, verbose=0)
    preds      = (preds > 0.5).astype(int).reshape(xx.shape)

    # Plot
    own_ax = ax is None
    if own_ax:
        fig, ax = plt.subplots(figsize=(7, 6))

    ax.contourf(xx, yy, preds, cmap=plt.cm.RdYlBu, alpha=0.4)
    ax.scatter(X[y == 0, 0], X[y == 0, 1],
               c='steelblue', edgecolors='white', s=30, alpha=0.8, label='Class 0')
    ax.scatter(X[y == 1, 0], X[y == 1, 1],
               c='tomato',    edgecolors='white', s=30, alpha=0.8, label='Class 1')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('x₁')
    ax.set_ylabel('x₂')
    ax.legend(loc='upper right')

    if own_ax:
        plt.tight_layout()
        plt.show()


print('plot_decision_boundary() defined — ready to use.')

In [ ]:
# ── Side-by-side decision boundaries: Model 1 vs Model 2 ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

m1_acc = model_1.evaluate(X, y, verbose=0)[1]
m2_acc = model_2.evaluate(X, y, verbose=0)[1]

plot_decision_boundary(model_1, X, y,
    title=f'Model 1 — 1 Dense layer / SGD\nAccuracy: {m1_acc*100:.1f}%', ax=axes[0])
plot_decision_boundary(model_2, X, y,
    title=f'Model 2 — 2 Hidden layers / Adam\nAccuracy: {m2_acc*100:.1f}%', ax=axes[1])

plt.suptitle('Decision Boundary Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Observation: Model 1 draws a straight line — useless for circles.')
print('Model 2 starts to curve but still misses without activation functions in hidden layers.')

---
## 6️⃣ Incorporate Activation Functions (ReLU + Sigmoid)

In [ ]:
# ── Quick visual explanation of activation functions ──────────────────────────
z = np.linspace(-3, 3, 300)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(z, np.maximum(0, z), color='darkorange', linewidth=2)
axes[0].axhline(0, color='gray', linewidth=0.8, linestyle='--')
axes[0].axvline(0, color='gray', linewidth=0.8, linestyle='--')
axes[0].set_title('ReLU — f(z) = max(0, z)', fontweight='bold')
axes[0].set_xlabel('z')
axes[0].set_ylabel('f(z)')
axes[0].grid(alpha=0.3)
axes[0].annotate('Kills negatives\n(sparsity)', xy=(-1.5, 0.1), fontsize=10, color='steelblue')
axes[0].annotate('Passes positives\nthrough unchanged', xy=(0.5, 2.2), fontsize=10, color='darkorange')

axes[1].plot(z, 1 / (1 + np.exp(-z)), color='purple', linewidth=2)
axes[1].axhline(0.5, color='gray', linewidth=0.8, linestyle='--')
axes[1].set_title('Sigmoid — f(z) = 1 / (1 + e⁻ᶻ)', fontweight='bold')
axes[1].set_xlabel('z')
axes[1].set_ylabel('f(z)')
axes[1].grid(alpha=0.3)
axes[1].annotate('Squashes output\nto (0, 1)', xy=(-2.5, 0.6), fontsize=10, color='purple')
axes[1].annotate('Used in output layer\nfor binary classification', xy=(-2.9, 0.08), fontsize=9, color='gray')

plt.tight_layout()
plt.show()

In [ ]:
tf.random.set_seed(42)

# ── Model 3: hidden layers with ReLU, output with Sigmoid ─────────────────────
model_3 = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation='relu',    input_shape=(2,)),  # ReLU hidden
    tf.keras.layers.Dense(16, activation='relu'),                        # ReLU hidden
    tf.keras.layers.Dense(1,  activation='sigmoid')                     # Sigmoid output
], name='model_3_relu_sigmoid')

model_3.compile(
    loss='binary_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    metrics=['accuracy']
)

model_3.summary()

print()
print('Why ReLU in hidden layers?')
print('  - Fast to compute, no vanishing gradient for positive values')
print('  - Introduces non-linearity so the network can learn curved boundaries')
print()
print('Why Sigmoid in the output layer?')
print('  - Maps any real number to (0, 1) — interpretable as a probability')
print('  - Threshold at 0.5 → predicted class 0 or 1')

In [ ]:
history_3 = model_3.fit(
    X, y,
    epochs=100,
    batch_size=32,
    verbose=0
)

loss_3, acc_3 = model_3.evaluate(X, y, verbose=0)
print(f'Model 3 (ReLU + Sigmoid) — Loss: {loss_3:.4f}  |  Accuracy: {acc_3*100:.2f}%')

---
## 7️⃣ Split Data into Training and Testing Sets

In [ ]:
# ── 80/20 train-test split ────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f'Training set : {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Test set     : {X_test.shape[0]}  samples ({X_test.shape[0]/len(X)*100:.0f}%)')

# ── Visualize the split ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, data_X, data_y, split_name in [
    (axes[0], X_train, y_train, 'Training'),
    (axes[1], X_test,  y_test,  'Test')
]:
    ax.scatter(data_X[data_y == 0, 0], data_X[data_y == 0, 1],
               c='steelblue', label='Class 0', alpha=0.7, edgecolors='white', s=35)
    ax.scatter(data_X[data_y == 1, 0], data_X[data_y == 1, 1],
               c='tomato',    label='Class 1', alpha=0.7, edgecolors='white', s=35)
    ax.set_title(f'{split_name} Set ({len(data_X)} samples)', fontweight='bold')
    ax.set_xlabel('x₁')
    ax.set_ylabel('x₂')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
tf.random.set_seed(42)

# ── Final model: deeper architecture, trained on X_train only ─────────────────
model_final = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(2,)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1,  activation='sigmoid')
], name='model_final')

model_final.compile(
    loss='binary_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    metrics=['accuracy']
)

history_final = model_final.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_test, y_test),   # monitor test performance each epoch
    verbose=0
)

train_loss, train_acc = model_final.evaluate(X_train, y_train, verbose=0)
test_loss,  test_acc  = model_final.evaluate(X_test,  y_test,  verbose=0)

print(f'Final model — Train: loss={train_loss:.4f}, acc={train_acc*100:.2f}%')
print(f'Final model — Test : loss={test_loss:.4f},  acc={test_acc*100:.2f}%')

---
## 8️⃣ Evaluate & Visualize Final Model Performance

In [ ]:
# ── Training curves (final model) ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history_final.history['loss']) + 1)

# Loss
axes[0].plot(epochs_range, history_final.history['loss'],     label='Train Loss',      marker='o', markevery=10)
axes[0].plot(epochs_range, history_final.history['val_loss'], label='Validation Loss', marker='s', markevery=10, linestyle='--')
axes[0].set_title('Final Model — Loss over Epochs', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Cross-Entropy')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, history_final.history['accuracy'],     label='Train Accuracy',      marker='o', markevery=10)
axes[1].plot(epochs_range, history_final.history['val_accuracy'], label='Validation Accuracy', marker='s', markevery=10, linestyle='--')
axes[1].set_title('Final Model — Accuracy over Epochs', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Decision boundaries: train set vs test set ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

plot_decision_boundary(
    model_final, X_train, y_train,
    title=f'Final Model — Training Set\nAcc: {train_acc*100:.1f}%  Loss: {train_loss:.4f}',
    ax=axes[0]
)
plot_decision_boundary(
    model_final, X_test, y_test,
    title=f'Final Model — Test Set\nAcc: {test_acc*100:.1f}%  Loss: {test_loss:.4f}',
    ax=axes[1]
)

plt.suptitle('Final Model Decision Boundary', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Full model comparison summary ─────────────────────────────────────────────
summary_data = {
    'Model': ['Model 1\n1 layer / SGD', 'Model 2\n2 layers / Adam\nNo activation', 'Model 3\n2 layers / ReLU+Sigmoid', 'Final\n3 layers / ReLU+Sigmoid'],
    'Accuracy (%)': [
        model_1.evaluate(X, y, verbose=0)[1] * 100,
        model_2.evaluate(X, y, verbose=0)[1] * 100,
        model_3.evaluate(X, y, verbose=0)[1] * 100,
        test_acc * 100
    ]
}

colors_bar = ['#d9534f', '#f0ad4e', '#5bc0de', '#5cb85c']
fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(summary_data['Model'], summary_data['Accuracy (%)'],
              color=colors_bar, edgecolor='white', width=0.5)
ax.set_ylim(40, 105)
ax.axhline(50, color='gray', linestyle='--', linewidth=1, label='Random baseline (50%)')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
ax.legend()
for bar, acc in zip(bars, summary_data['Accuracy (%)']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
            f'{acc:.1f}%', ha='center', fontweight='bold', fontsize=11)
plt.tight_layout()
plt.show()

---
## 9️⃣ Summary of Key Takeaways

### What we learned

**1. Classification types matter for architecture choices.**  
Binary classification uses 1 output neuron + Sigmoid; multi-class uses N neurons + Softmax; multi-label uses N neurons + Sigmoid per label. Choosing wrong leads to incorrect loss functions and uninterpretable outputs.

**2. Visualizing data is the first step — always.**  
The scatter plot immediately revealed that the two classes form concentric rings. A linear model is structurally incapable of solving this problem regardless of training duration. Knowing the data shape guided every architectural decision.

**3. A single linear layer cannot learn non-linear boundaries.**  
Model 1 (1 dense layer, no activation, SGD) achieved ~50% accuracy — random chance. Without depth and non-linearity, the network can only draw a straight line.

**4. Activation functions are essential, not optional.**  
Adding ReLU to hidden layers (Model 3) was the single biggest jump in accuracy. Without it, stacking more dense layers just produces another linear transformation. ReLU introduces the non-linearity that allows the network to bend and curve its decision boundary.

**5. The optimizer and learning rate have a large effect.**  
Switching from SGD to Adam dramatically improved convergence speed. Adam adapts per-parameter learning rates automatically, making it far more robust than vanilla SGD for most neural network tasks.

**6. Train/test splitting reveals overfitting.**  
Training on all data and evaluating on the same data is misleading. The proper pipeline is: split first → train on training set → evaluate on held-out test set. Similar train and test accuracy means the model generalizes; a large gap means overfitting.

**7. Decision boundary plots are the best debugging tool.**  
Accuracy numbers alone don't show *why* a model fails. The decision boundary plots instantly showed that Model 1 draws a straight line (wrong shape), while the final model learned a circular boundary that matches the data geometry.

### Hyperparameter tuning checklist for classification
| Hyperparameter | What to try |
|---|---|
| Number of layers | Start with 2–3 hidden; increase if underfitting |
| Neurons per layer | Powers of 2: 16, 32, 64, 128 |
| Hidden activation | **ReLU** (default), Leaky ReLU, ELU |
| Output activation | **Sigmoid** (binary), Softmax (multi-class) |
| Optimizer | **Adam** (default), SGD with momentum |
| Learning rate | Start at 0.01; reduce if loss oscillates |
| Epochs | Monitor val_loss and stop early when it plateaus |
| Batch size | 32–128; larger = faster but noisier gradients |